# Intelligent Player Churn Prediction
**Objective:** Build a machine learning model to predict player churn using historical gameplay data.
**Models Used:** Logistic Regression (Baseline), Random Forest Classifier

In [2]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

# Loading dataset
df = pd.read_csv("player_data.csv")
df.head()

,PlayerID,Age,Gender,Location,GameGenre,PlayTimeHours,InGamePurchases,GameDifficulty,SessionsPerWeek,AvgSessionDurationMinutes,PlayerLevel,AchievementsUnlocked,EngagementLevel
0,9000,43,Male,Other,Strategy,16.271119,0,Medium,6,108,79,25,Medium
1,9001,29,Female,USA,Strategy,5.525961,0,Medium,5,144,11,10,Medium
2,9002,22,Female,USA,Sports,8.223755,0,Easy,16,142,35,41,High
3,9003,35,Male,USA,Action,5.265351,1,Easy,9,85,57,47,Medium
4,9004,33,Male,Europe,Action,15.531945,0,Medium,2,131,95,37,Medium


### Feature Engineering
The original dataset uses a categorical 'EngagementLevel' column. To frame this as a churn prediction problem, we enginner a binary 'Churn' target variable:
- 'Low' Engagement = 1 (Churned)
- 'Medium' and 'High' Engagement = 0 (Not churned)

In [3]:
# Creating binary target variable
df['Churn'] = df['EngagementLevel'].apply(lambda x: 1 if x == 'Low' else 0)

# Dropping unused columns and old target variable
X = df.drop(columns=['PlayerID', 'EngagementLevel', 'Churn'])
y = df['Churn']

print(f"Dataset shape: {X.shape}")
print(f"Chrun distribution: {y.value_counts(normalize=True)*100}")

Dataset shape: (40034, 11)
Chrun distribution: Churn
0    74.21192
1    25.78808
Name: proportion, dtype: float64


In [ ]:
# Seaparating features by type so we can process them differently
numeric_features = ['Age', 'PlayTimeHours', 'InGamePurchases', 'SessionsPerWeek', 'AvgSessionDurationMinutes', 'PlayerLevel', 'AchievementsUnlocked']
categorical_features = ['Gender', 'Location', 'GameGenre', 'GameDifficulty']

# Building the preprocessor rules
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

print("Preprocessor ready!")

Preprocessor ready!


In [5]:
# Splitting data, 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Baseline model: Logistic Regression
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000))
])
print("Training logistic regression...")
lr_pipeline.fit(X_train, y_train)

# Primary model: random forest
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1))
])
print("Training random forest...")
rf_pipeline.fit(X_train, y_train)

print("Both models trained successfully!")

Training logistic regression...
Training random forest...
Both models trained successfully!


In [7]:
import joblib
from sklearn.metrics import roc_auc_score
from sklearn.metrics import recall_score
from sklearn.metrics import accuracy_score
# Evaluation function to print metrics
def evaluate_model(model, name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    print(f"\n=== {name} ===")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
    print(f"Precision: {precision_score(y_test, y_pred):.3f}")
    print(f"Recall: {recall_score(y_test, y_pred):.3f}")
    print(f"AUC Score: {roc_auc_score(y_test, y_pred):.3f}")

evaluate_model(lr_pipeline, "Logistic Regression")
evaluate_model(rf_pipeline, "Random Forest")

joblib.dump(rf_pipeline, 'churn_model.pkl')
print("\nModel successfully saved as 'churn_model.pkl'!")


=== Logistic Regression ===
Accuracy: 0.824
Precision: 0.619
Recall: 0.850
AUC Score: 0.832

=== Random Forest ===
Accuracy: 0.939
Precision: 0.908
Recall: 0.853
AUC Score: 0.911

Model successfully saved as 'churn_model.pkl'!
